# Sierra to Workday Time-Off Sync

Step 1 setup:
- Sierra source query
- Workday token generation


In [ ]:
SIERRA_TIMEOFF_QUERY = """
SELECT
    HP.INTERNAL_NUM        AS TimeKeeper,
    HP.EMPLOYEE_CODE       AS WorkdayId,
    TT.TOBILL_HRS          AS hrs,
    year(TRAN_DATE)        AS WorkedYear,
    TRAN_DATE              AS timecard_worked_date,
    PERIOD                 AS timecard_worked_cal_period,
    date_format(
        to_date(cast(PERIOD as string), 'yyyyMM'),
        'MMyy'
    )                      AS WP_PPYY,
    POST_DATE              AS Timecard_post_date,
    HC.CLIENT_CODE,
    MATTER_CODE,
    CASE
        WHEN min(POST_DATE) OVER (PARTITION BY HP.INTERNAL_NUM, TRAN_DATE) = POST_DATE
        THEN 'I'
        ELSE 'U'
    END AS InsertUpdate,
    HP.`POSITION`          AS jobtitle
FROM sc_bronze.HBM_MATTER M
JOIN sc_bronze.TAT_TIME TT
    ON M.MATTER_UNO = TT.MATTER_UNO
JOIN sc_bronze.HBM_CLIENT HC
    ON HC.CLIENT_UNO = M.CLIENT_UNO
JOIN sc_bronze.HBM_PERSNL HP
    ON HP.EMPL_UNO = TT.TK_EMPL_UNO
JOIN sc_bronze.HBL_DEPT HD
    ON HD.DEPT_CODE = HP.DEPT
JOIN sc_bronze.HBL_OFFICE HO
    ON HO.OFFC_CODE = HP.OFFC
WHERE MATTER_CODE IN (
        '8000000028','1000325429','8000000016','1000325434','1000086654'
      )
  AND year(TRAN_DATE) >= year(current_date()) - 1
  AND HP.`POSITION` IN ('Associate', 'Counsel')
  AND HO.OFFC_CODE IN (
        'AUS1','CHI1','DAL1','DEN1','HOU1','LAX1',
        'IPS1','NYC1','PIT1','SAT1','SFO1','STL1','WAS1'
      )
  AND lower(HP.`POSITION`) NOT LIKE '%partner%'
"""


def get_sierra_timeoff_query() -> str:
    return SIERRA_TIMEOFF_QUERY


def fetch_sierra_timeoff_df(spark):
    query = get_sierra_timeoff_query()
    return spark.sql(query)


In [ ]:
import requests


def get_workday_access_token(
    token_url: str,
    refresh_token: str,
    authorization: str,
    timeout: int = 30,
) -> str:
    payload = {
        "grant_type": "refresh_token",
        "refresh_token": refresh_token,
    }
    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Authorization": authorization,
    }
    response = requests.post(token_url, data=payload, headers=headers, timeout=timeout)
    response.raise_for_status()
    body = response.json()
    access_token = body.get("access_token")
    if not access_token:
        raise ValueError("access_token missing from token response")
    return access_token


def build_workday_headers(access_token: str) -> dict:
    return {"Authorization": f"Bearer {access_token}"}
